In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages


# ============================================================
# SETTINGS
# ============================================================

ROOT = Path(".").resolve()

FEATURE_SETS = [30, 20, 12]

# Colors
BACKGROUND_COLOR = "#BDBDBD"

RAW_COLOR = "#E67E22"       # without calibration
CALIBRATED_COLOR = "#7B4AB3"  # with calibration


# ============================================================
# LOAD SUPERVISED DATA
# ============================================================

def load_supervised_data(root, transition, standardized=False):

    folder = root / "SUPERVISED" / transition

    suffix = "_STD" if standardized else ""

    mean_path = folder / f"{transition}{suffix}_curve_mean.csv"

    if not mean_path.exists():
        raise FileNotFoundError(mean_path)

    return pd.read_csv(mean_path)


# ============================================================
# FILTER SUPERVISED CURVE
# ============================================================

def filter_supervised_curve(
    df,
    feature_set,
    model,
    x_column,
    calibration=None,
):
    """
    Select one supervised model and feature set.

    calibration:
        None          -> all calibration values
        "calibrated"  -> calibrated only
        "uncalibrated" -> uncalibrated only
    """

    result = df[
        (df["feature_set"] == feature_set)
        & (df["model"] == model)
    ].copy()

    if calibration is not None and "calibration" in result.columns:

        result = result[
            result["calibration"] == calibration
        ].copy()

    return result.sort_values(x_column)


# ============================================================
# GET SUPERVISED MODELS
# ============================================================

def get_supervised_models(raw_df):

    return sorted(
        raw_df["model"]
        .dropna()
        .unique()
    )


# ============================================================
# GET CALIBRATION VALUES
# ============================================================

def get_calibration_values(df):

    if "calibration" not in df.columns:
        raise ValueError(
            "Column 'calibration' was not found in supervised data."
        )

    values = (
        df["calibration"]
        .dropna()
        .unique()
    )

    print("Available calibration values:")
    print(values)

    return values


# ============================================================
# PLOT SUPERVISED BACKGROUND
# ============================================================

def plot_supervised_background(
    ax,
    df,
    feature_set,
    selected_model,
    x_column,
):
    """
    Plot all OTHER supervised models in gray.
    """

    models = (
        df["model"]
        .dropna()
        .unique()
    )

    for model in models:

        # Do not plot the model currently being compared
        if model == selected_model:
            continue

        curve = filter_supervised_curve(
            df=df,
            feature_set=feature_set,
            model=model,
            x_column=x_column,
            calibration=None,
        )

        if curve.empty:
            continue

        ax.errorbar(
            curve[x_column],
            curve["mean"],
            yerr=curve["mean_err"],
            fmt="o-",
            color=BACKGROUND_COLOR,
            ecolor=BACKGROUND_COLOR,
            markersize=2,
            linewidth=0.9,
            capsize=1.5,
            elinewidth=0.6,
            alpha=0.45,
            zorder=1,
        )


# ============================================================
# PLOT ONE CALIBRATION COMPARISON PANEL
# ============================================================

def plot_calibration_panel(
    ax,
    supervised_df,
    feature_set,
    model,
    x_column,
):
    """
    One panel:

        gray       = other supervised models
        orange     = selected model, uncalibrated
        purple     = selected model, calibrated
    """

    # ========================================================
    # OTHER SUPERVISED MODELS — GRAY
    # ========================================================

    plot_supervised_background(
        ax=ax,
        df=supervised_df,
        feature_set=feature_set,
        selected_model=model,
        x_column=x_column,
    )

    # ========================================================
    # SELECTED MODEL — WITHOUT CALIBRATION
    # ========================================================

    uncalibrated_curve = filter_supervised_curve(
        df=supervised_df,
        feature_set=feature_set,
        model=model,
        x_column=x_column,
        calibration="uncalibrated",
    )

    if not uncalibrated_curve.empty:

        ax.errorbar(
            uncalibrated_curve[x_column],
            uncalibrated_curve["mean"],
            yerr=uncalibrated_curve["mean_err"],
            fmt="o-",
            color=RAW_COLOR,
            ecolor=RAW_COLOR,
            markersize=3.5,
            linewidth=2.0,
            capsize=2.5,
            elinewidth=0.9,
            label="bez kalibracji",
            zorder=4,
        )

    # ========================================================
    # SELECTED MODEL — WITH CALIBRATION
    # ========================================================

    calibrated_curve = filter_supervised_curve(
        df=supervised_df,
        feature_set=feature_set,
        model=model,
        x_column=x_column,
        calibration="calibrated",
    )

    if not calibrated_curve.empty:

        ax.errorbar(
            calibrated_curve[x_column],
            calibrated_curve["mean"],
            yerr=calibrated_curve["mean_err"],
            fmt="o-",
            color=CALIBRATED_COLOR,
            ecolor=CALIBRATED_COLOR,
            markersize=3.5,
            linewidth=2.0,
            capsize=2.5,
            elinewidth=0.9,
            label="z kalibracją",
            zorder=5,
        )

    # ========================================================
    # p = 0.5
    # ========================================================

    ax.axhline(
        0.5,
        color="0.65",
        linestyle=":",
        linewidth=0.8,
        zorder=0,
    )

    # ========================================================
    # PANEL TITLE
    # ========================================================

    ax.set_title(
        f"{feature_set} cech",
        fontsize=13,
        fontweight="bold",
        pad=10,
    )

    # ========================================================
    # AXES
    # ========================================================

    ax.set_xlabel(
        r"$\Delta$" if x_column == "Delta" else r"$K_0$",
        fontsize=11,
    )

    ax.set_ylim(
        -0.05,
        1.05,
    )

    ax.grid(
        alpha=0.25,
        linewidth=0.6,
    )

    ax.tick_params(
        axis="both",
        labelsize=9,
    )


# ============================================================
# CREATE ONE FIGURE FOR ONE MODEL
# ============================================================

def make_calibration_figure(
    transition,
    model,
    supervised_df,
    x_column,
):
    """
    Create one figure for one supervised model.

    Three panels:
        30 features
        20 features
        12 features
    """

    fig, axes = plt.subplots(
        nrows=1,
        ncols=3,
        figsize=(13.5, 4.5),
        sharey=True,
    )

    # ========================================================
    # PANELS
    # ========================================================

    for ax, feature_set in zip(
        axes,
        FEATURE_SETS,
    ):

        plot_calibration_panel(
            ax=ax,
            supervised_df=supervised_df,
            feature_set=feature_set,
            model=model,
            x_column=x_column,
        )

    # ========================================================
    # Y LABEL
    # ========================================================

    axes[0].set_ylabel(
        r"$\langle \Pr(d_j(\Delta)\in C_b)\rangle$",
        fontsize=11,
    )

    # ========================================================
    # LEGEND
    # ========================================================

    handles, labels = axes[0].get_legend_handles_labels()

    if handles:

        fig.legend(
            handles,
            labels,
            loc="lower center",
            bbox_to_anchor=(0.5, -0.025),
            ncol=2,
            frameon=False,
            fontsize=9,
        )

    # ========================================================
    # TITLE
    # ========================================================

    fig.suptitle(
        f"{transition} — {model}",
        fontsize=14,
        fontweight="bold",
        y=1.02,
    )

    # ========================================================
    # LAYOUT
    # ========================================================

    plt.tight_layout(
        rect=[
            0.0,
            0.08,
            1.0,
            0.96,
        ]
    )

    return fig, axes


# ============================================================
# CREATE ALL FIGURES FOR ONE TRANSITION
# ============================================================

def make_all_calibration_figures(
    transition,
    pdf,
):
    """
    Create one figure for every supervised model.
    """

    # ========================================================
    # X AXIS
    # ========================================================

    if transition == "AC":
        x_column = "K0"

    elif transition == "BCb":
        x_column = "Delta"
        
    elif transition == "AB":
        x_column = "Delta"

    else:
        raise ValueError(
            f"Unknown transition: {transition}"
        )

    # ========================================================
    # LOAD DATA
    # ========================================================

    supervised_df = load_supervised_data(
        ROOT,
        transition,
        standardized=False,
    )

    # ========================================================
    # CHECK CALIBRATION VALUES
    # ========================================================

    get_calibration_values(
        supervised_df
    )

    # ========================================================
    # GET MODELS
    # ========================================================

    models = get_supervised_models(
        supervised_df
    )

    print("=" * 70)
    print(f"TRANSITION: {transition}")
    print("=" * 70)
    print("Supervised models:")
    print(models)
    print()

    # ========================================================
    # CREATE ONE FIGURE PER MODEL
    # ========================================================

    for model in models:

        print(
            f"Plotting: {transition} — {model}"
        )

        fig, axes = make_calibration_figure(
            transition=transition,
            model=model,
            supervised_df=supervised_df,
            x_column=x_column,
        )

        pdf.savefig(
            fig,
            bbox_inches="tight",
        )

        plt.close(fig)


# ============================================================
# RUN
# ============================================================

output_dir = ROOT / "figures_calibration_comparison"

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

pdf_path = (
    output_dir
    / "supervised_calibration_comparison.pdf"
)

with PdfPages(pdf_path) as pdf:

    make_all_calibration_figures(
        "AC",
        pdf,
    )

    make_all_calibration_figures(
        "BCb",
        pdf,
    )
    make_all_calibration_figures(
        "AB",
        pdf,
    )

print(
    f"Saved: {pdf_path}"
)

Available calibration values:
['calibrated' 'uncalibrated']
TRANSITION: AC
Supervised models:
['Decision Tree', 'Gradient Boosted Trees', 'Logistic Regression', 'Logistic Regression C = 0.00175', 'Neural Network', 'Random Forest', 'SVM (RBF)', 'kNN', 'kNN ball_tree', 'kNN kd_tree']

Plotting: AC — Decision Tree
Plotting: AC — Gradient Boosted Trees
Plotting: AC — Logistic Regression
Plotting: AC — Logistic Regression C = 0.00175
Plotting: AC — Neural Network
Plotting: AC — Random Forest
Plotting: AC — SVM (RBF)
Plotting: AC — kNN
Plotting: AC — kNN ball_tree
Plotting: AC — kNN kd_tree
Available calibration values:
['calibrated' 'uncalibrated']
TRANSITION: BCb
Supervised models:
['Decision Tree', 'Gradient Boosted Trees', 'Logistic Regression', 'Logistic Regression C = 0.00175', 'Neural Network', 'Random Forest', 'SVM (RBF)', 'kNN', 'kNN ball_tree', 'kNN kd_tree']

Plotting: BCb — Decision Tree
Plotting: BCb — Gradient Boosted Trees
Plotting: BCb — Logistic Regression
Plotting: BCb — Lo